### 1. Install dependency

In [1]:

!pip install -q transformers torch accelerate openpyxl


### 2. Load your Excel news dataset

In [3]:
import pandas as pd

news_path = "../data/raw/apple_news_data.xlsx"   # change if needed
news_df = pd.read_excel(news_path)

print(news_df.columns)
news_df.head()

Index(['date', 'title', 'content', 'link', 'symbols', 'tags',
       'sentiment_polarity', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos',
       'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25',
       'Unnamed: 26'],
      dtype='object')


,date,title,content,link,symbols,tags,sentiment_polarity,sentiment_neg,sentiment_neu,sentiment_pos,...,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,2024-11-27T16:39:00+00:00,Berkshire Stock Hits Record Even as Company Re...,"Warren Buffett鈥檚 caution, his advancing age, a...",https://finance.yahoo.com/m/f5df3aa4-364b-31d6...,"0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-26T00:00:00+00:00,What Is a Stock Market Index?,What Is a Stock Market Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, MSFT.US",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-11-26T00:00:00+00:00,"Could Investing $1,000 in Apple Make You a Mil...","Could Investing $1,000 in Apple Make You a Mil...",https://www.fool.com/investing/2024/11/26/coul...,AAPL.US,NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-11-26T00:00:00+00:00,Dow Jones Industrial Average,Dow Jones Industrial Average,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMGN.US, AMZN.US, CSCO.US, GOOG.US, G...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-11-26T00:00:00+00:00,What Is the S&P 500 Index?,What Is the S&P 500 Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, GOOG.US, GOOGL.US, META.US, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Specify the text and date columns

In [4]:
TEXT_COL = "content"   
DATE_COL = "date"      

# Drop rows without text
news_df = news_df.dropna(subset=[TEXT_COL]).copy()

# Convert date column to datetime
news_df[DATE_COL] = pd.to_datetime(news_df[DATE_COL], errors="coerce")
news_df = news_df.dropna(subset=[DATE_COL]).copy()

# Convert to daily date for merging with price data later
news_df["merge_date"] = news_df[DATE_COL].dt.date

news_df.head()

,date,title,content,link,symbols,tags,sentiment_polarity,sentiment_neg,sentiment_neu,sentiment_pos,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,merge_date
0,2024-11-27 16:39:00+00:00,Berkshire Stock Hits Record Even as Company Re...,"Warren Buffett鈥檚 caution, his advancing age, a...",https://finance.yahoo.com/m/f5df3aa4-364b-31d6...,"0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-27
1,2024-11-26 00:00:00+00:00,What Is a Stock Market Index?,What Is a Stock Market Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, MSFT.US",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-26
2,2024-11-26 00:00:00+00:00,"Could Investing $1,000 in Apple Make You a Mil...","Could Investing $1,000 in Apple Make You a Mil...",https://www.fool.com/investing/2024/11/26/coul...,AAPL.US,NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-26
3,2024-11-26 00:00:00+00:00,Dow Jones Industrial Average,Dow Jones Industrial Average,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMGN.US, AMZN.US, CSCO.US, GOOG.US, G...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-26
4,2024-11-26 00:00:00+00:00,What Is the S&P 500 Index?,What Is the S&P 500 Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, GOOG.US, GOOGL.US, META.US, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-11-26


### 4. Load the FinBERT model

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# Label mapping for ProsusAI/finbert:
# 0 = neutral, 1 = positive, 2 = negative
id2label = {0: "neutral", 1: "positive", 2: "negative"}

c:\Users\Lenovo\anaconda3\envs\MLE\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


c:\Users\Lenovo\anaconda3\envs\MLE\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downlo

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


### 5. Define a batch FinBERT inference function

In [6]:
import numpy as np
from tqdm.auto import tqdm

def finbert_sentiment_batch(texts, batch_size=16, max_length=256):
    """
    Run FinBERT sentiment classification in batches.
    
    Args:
        texts (list[str]): list of news text
        batch_size (int): batch size for inference
        max_length (int): max sequence length
        
    Returns:
        (neg, neu, pos): three numpy arrays of probabilities
    """
    
    all_neg, all_neu, all_pos = [], [], []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize the batch
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        enc = {k: v.to(device) for k, v in enc.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**enc)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        
        # Order: [neutral, positive, negative]
        neu = probs[:, 0]
        pos = probs[:, 1]
        neg = probs[:, 2]
        
        all_neg.append(neg)
        all_neu.append(neu)
        all_pos.append(pos)
    
    # Concatenate all outputs
    all_neg = np.concatenate(all_neg)
    all_neu = np.concatenate(all_neu)
    all_pos = np.concatenate(all_pos)
    
    return all_neg, all_neu, all_pos

### 6. Run FinBERT on all texts

In [7]:
texts = news_df[TEXT_COL].astype(str).tolist()

neg, neu, pos = finbert_sentiment_batch(texts, batch_size=32, max_length=256)

news_df["finbert_neg"] = neg
news_df["finbert_neu"] = neu
news_df["finbert_pos"] = pos

# Simple polarity score (positive minus negative)
news_df["finbert_polarity"] = news_df["finbert_pos"] - news_df["finbert_neg"]

# Optional: most likely sentiment label
labels_idx = np.stack([neu, pos, neg], axis=1).argmax(axis=1)
news_df["finbert_label"] = [id2label[x] for x in labels_idx]

news_df.head()

100%|██████████| 929/929 [1:19:58<00:00,  5.16s/it]


,date,title,content,link,symbols,tags,sentiment_polarity,sentiment_neg,sentiment_neu,sentiment_pos,...,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,merge_date,finbert_neg,finbert_neu,finbert_pos,finbert_polarity,finbert_label
0,2024-11-27 16:39:00+00:00,Berkshire Stock Hits Record Even as Company Re...,"Warren Buffett鈥檚 caution, his advancing age, a...",https://finance.yahoo.com/m/f5df3aa4-364b-31d6...,"0R2V.IL, AAPL.BA, AAPL.MX, AAPL.NEO, AAPL.SN, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,2024-11-27,0.737745,0.021739,0.240516,-0.497229,negative
1,2024-11-26 00:00:00+00:00,What Is a Stock Market Index?,What Is a Stock Market Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, MSFT.US",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,2024-11-26,0.905479,0.025893,0.068628,-0.836851,negative
2,2024-11-26 00:00:00+00:00,"Could Investing $1,000 in Apple Make You a Mil...","Could Investing $1,000 in Apple Make You a Mil...",https://www.fool.com/investing/2024/11/26/coul...,AAPL.US,NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,2024-11-26,0.917303,0.064256,0.018441,-0.898862,negative
3,2024-11-26 00:00:00+00:00,Dow Jones Industrial Average,Dow Jones Industrial Average,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMGN.US, AMZN.US, CSCO.US, GOOG.US, G...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,2024-11-26,0.894662,0.047523,0.057814,-0.836848,negative
4,2024-11-26 00:00:00+00:00,What Is the S&P 500 Index?,What Is the S&P 500 Index?,https://www.fool.com/investing/stock-market/in...,"AAPL.US, AMZN.US, GOOG.US, GOOGL.US, META.US, ...",NaN,0,0,1,0,...,NaN,NaN,NaN,NaN,2024-11-26,0.896063,0.025314,0.078623,-0.817440,negative


### 7. Aggregate to daily-level sentiment

In [8]:
sent_daily = (
    news_df
    .groupby("merge_date")
    .agg(
        avg_sentiment_neg=("finbert_neg", "mean"),
        avg_sentiment_neu=("finbert_neu", "mean"),
        avg_sentiment_pos=("finbert_pos", "mean"),
        avg_sentiment_polarity=("finbert_polarity", "mean"),
        daily_news_count=(TEXT_COL, "count")
    )
    .reset_index()
)

sent_daily.head()

,merge_date,avg_sentiment_neg,avg_sentiment_neu,avg_sentiment_pos,avg_sentiment_polarity,daily_news_count
0,2016-02-19,0.295164,0.697282,0.007555,-0.287609,1
1,2017-10-05,0.944042,0.036873,0.019085,-0.924957,1
2,2017-11-27,0.944844,0.035581,0.019575,-0.925270,1
3,2017-11-30,0.748792,0.241450,0.009758,-0.739034,1
4,2018-01-31,0.944325,0.036840,0.018835,-0.925490,1


### 8. Save FinBERT Sentiment as CSV into data/processed/

In [ ]:
import os

# Create directory if it doesn't exist
save_dir = "../data/processed"
os.makedirs(save_dir, exist_ok=True)

# Output filepath
output_path = os.path.join(save_dir, "apple_news_data.csv")

# Save CSV
sent_daily.to_csv(output_path, index=False)

print("Saved to:", output_path)


Saved to: ../data/processed\finbert_sentiment_daily.csv
